# voiceai — Colab Training (1-Klick)

**So benutzt du das:**
1. Oben im Menü: **Runtime → Change runtime type → T4 GPU** (gratis) und speichern.
2. Trage in Zelle 3 deinen Hugging-Face-Token ein (https://huggingface.co/settings/tokens).
3. Menü: **Runtime → Run all**. Fertig.

Zelle 4 = Smoke-Test (validiert ganzen Stack auf CPU, ~5 min, $0).
Zelle 5 = GPU forward + backward mit echtem Mimi-Codec (~2 min).

Wenn alles grün durchläuft, ist der Stack bereit für einen langen, bezahlten Lauf.

## 1) Repo klonen + installieren

In [ ]:
!git clone https://github.com/chukfinley/voiceai.git
%cd voiceai
!pip install -q -e '.[train,dev]'
print('install done')

## 2) GPU prüfen

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Keine GPU! Runtime -> Change runtime type -> T4 GPU')
print('GPU:', torch.cuda.get_device_name(0))

## 3) Hugging-Face-Token (hier eintragen)

In [ ]:
import os
os.environ['HF_TOKEN'] = 'PASTE_YOUR_HF_TOKEN_HERE'  # <-- nur DIESE Zeile aendern
assert os.environ['HF_TOKEN'].startswith('hf_'), 'HF-Token noch nicht eingetragen!'
print('token gesetzt')

## 4) Smoke-Test — validiert den ganzen Stack (tiny model, kein Download)

In [ ]:
!python scripts/smoke_test.py

## 5) GPU forward + backward (echter Mimi-Codec, tiny model)

Lädt den echten Mimi-Codec + ein winziges Backbone auf die GPU, encodet
Dummy-Audio und macht einen forward + backward. Beweist, dass der GPU-Pfad
ohne NaN/OOM läuft. Braucht keine Datendateien.

In [ ]:
import torch
from voiceai.model.voiceai_lm import VoiceAIConfig, VoiceAILM
from voiceai.model.mimi_utils import load_mimi, mimi_encode

cfg = VoiceAIConfig(
    backbone='hf-internal-testing/tiny-random-LlamaForCausalLM',
    freeze_backbone=True, train_text=True, train_asst_audio=True,
    dtype='bfloat16',
)
model = VoiceAILM(cfg).cuda()
mimi = load_mimi(device='cuda', dtype=torch.bfloat16)
print(f'trainable: {model.trainable_param_count()/1e6:.1f}M / {model.total_param_count()/1e6:.1f}M')

audio = torch.randn(1, 1, 24000 * 2, device='cuda', dtype=torch.bfloat16)
codes = mimi_encode(mimi, audio)
text_ids = torch.zeros((1, codes.shape[2]), dtype=torch.long, device='cuda')
out = model(text_ids=text_ids, user_audio_codes=codes)
loss = out['text_logits'].float().pow(2).mean()
loss.backward()
print('text_logits:', tuple(out['text_logits'].shape))
print('backward OK, GPU mem:', f'{torch.cuda.memory_allocated()/1e9:.2f} GB')

## Fertig

Lief alles grün durch? Dann ist der Stack bereit.

**Nächster Schritt für einen echten Lauf** (nicht hier — Colab trennt nach 12h):
- ersetze `--backbone` durch `Qwen/Qwen3.5-0.8B`
- entferne `--smoke`, setze `--steps 30000`
- Checkpoints zu Hugging-Face-Hub pushen (Colab kann Drive in der VS-Code-Extension nicht mounten)

Für lange Läufe ist eine SSH-GPU-Box (Runpod RTX 3090) sauberer — siehe `HOW_TO_TRAIN.md`.